In [ ]:
# notebooks/main.ipynb
"""
# Projet de Prédiction Immobilière avec K-means
## ECE Paris - 2025

### 1. Importation des librairies
"""
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_collection import DataCollector
from modeling import PricePredictor

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)

"""
### 2. Collecte des données
"""
collector = DataCollector()
df_raw = collector.collect_paris_data(pages=50)
df_enriched = collector.add_external_features(df_raw)

print(f"Données collectées : {len(df_raw)} annonces")
print(f"Colonnes : {df_raw.columns.tolist()}")

"""
### 3. Exploration des données
"""
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Distribution des prix
axes[0, 0].hist(df_enriched['prix'], bins=30, edgecolor='black')
axes[0, 0].set_title('Distribution des Prix')
axes[0, 0].set_xlabel('Prix (€)')

# Prix au m2 par arrondissement
arrondissement_price = df_enriched.groupby('arrondissement')['prix_m2'].mean()
axes[0, 1].bar(arrondissement_price.index, arrondissement_price.values)
axes[0, 1].set_title('Prix au m² moyen par Arrondissement')
axes[0, 1].set_xlabel('Arrondissement')

# Surface vs Prix
axes[0, 2].scatter(df_enriched['surface'], df_enriched['prix'], alpha=0.6)
axes[0, 2].set_title('Surface vs Prix')
axes[0, 2].set_xlabel('Surface (m²)')
axes[0, 2].set_ylabel('Prix (€)')

plt.tight_layout()
plt.show()

"""
### 4. Pipeline complet
"""
predictor = PricePredictor()
results = predictor.create_prediction_pipeline(df_enriched)

# Affichage des résultats
print("\n=== RÉSULTATS DES MODÈLES ===")
for model_name, metrics in results['results'].items():
    print(f"\n{model_name.upper()}:")
    print(f"  MAE: {metrics['mae']:,.2f} €")
    print(f"  RMSE: {metrics['rmse']:,.2f} €")
    print(f"  R²: {metrics['r2']:.3f}")

print("\n=== ANALYSE DES CLUSTERS ===")
for cluster_id, analysis in results['cluster_analysis'].items():
    print(f"\nCluster {cluster_id}:")
    print(f"  Nombre d'annonces: {analysis['taille']}")
    print(f"  Prix moyen: {analysis['prix_moyen']:,.2f} €")
    print(f"  Prix/m² moyen: {analysis['prix_m2_moyen']:,.2f} €/m²")
    print(f"  Surface moyenne: {analysis['surface_moyenne']:.1f} m²")

"""
### 5. Visualisation des clusters
"""
fig = plt.figure(figsize=(15, 5))

# 1. Clusters dans l'espace prix/surface
ax1 = fig.add_subplot(131)
scatter = ax1.scatter(df_enriched['surface'], df_enriched['prix'], 
                     c=results['labels'], cmap='viridis', alpha=0.7)
ax1.set_xlabel('Surface (m²)')
ax1.set_ylabel('Prix (€)')
ax1.set_title('Segmentation K-means: Surface vs Prix')
plt.colorbar(scatter, ax=ax1, label='Cluster')

# 2. Répartition géographique
ax2 = fig.add_subplot(132)
arrondissement_cluster = pd.crosstab(df_enriched['arrondissement'], results['labels'])
arrondissement_cluster.plot(kind='bar', stacked=True, ax=ax2)
ax2.set_xlabel('Arrondissement')
ax2.set_ylabel('Nombre d\'annonces')
ax2.set_title('Répartition des Clusters par Arrondissement')
ax2.legend(title='Cluster')

# 3. Caractéristiques moyennes par cluster
ax3 = fig.add_subplot(133)
cluster_means = df_enriched.groupby(results['labels'])[['prix_m2', 'pieces', 'surface']].mean()
cluster_means.plot(kind='bar', ax=ax3)
ax3.set_xlabel('Cluster')
ax3.set_ylabel('Valeur moyenne')
ax3.set_title('Caractéristiques moyennes par Cluster')
ax3.legend(['Prix/m² (€)', 'Pièces', 'Surface (m²)'])

plt.tight_layout()
plt.show()

"""
### 6. Prédictions vs Réalité
"""
best_model_name = min(results['results'], 
                     key=lambda x: results['results'][x]['rmse'])
best_predictions = results['results'][best_model_name]['predictions']

plt.figure(figsize=(10, 6))
plt.scatter(results['test_data'][1], best_predictions, alpha=0.6)
plt.plot([results['test_data'][1].min(), results['test_data'][1].max()],
         [results['test_data'][1].min(), results['test_data'][1].max()],
         'r--', lw=2)
plt.xlabel('Prix Réel (€)')
plt.ylabel('Prix Prédit (€)')
plt.title(f'Prédictions vs Réalité - {best_model_name.upper()}')
plt.grid(True)

# Calcul et affichage des erreurs
errors = best_predictions - results['test_data'][1]
plt.figure(figsize=(10, 4))
plt.hist(errors, bins=30, edgecolor='black')
plt.xlabel('Erreur de prédiction (€)')
plt.ylabel('Fréquence')
plt.title('Distribution des erreurs de prédiction')
plt.axvline(x=0, color='r', linestyle='--')
plt.show()